In [3]:
import sqlite3
import pandas as pd 
import numpy as np 
from scipy import stats

con = sqlite3.connect('C:\\Users\\muham\\OneDrive\\Documents\\airtel telicom analysis\\data\\raw\\airtel.db')
def q(sql):
    return pd.read_sql_query(sql, con)

In [4]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [5]:
def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
    pool = (k1 + k2) / (n1 + n2)               # both groups as one
    se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
    z = (k1/n1 - k2/n2) / se                   # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [6]:
pre=q("""
SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;
""")
print(pre)

       g  customer_id            m
0      B            1   729.705882
1      A            2   256.590909
2      A            3  1914.133333
3      B            4   218.880000
4      B            5   980.926667
...   ..          ...          ...
21988  A        21996  6681.156863
21989  A        21997  1460.978571
21990  B        21998    69.021429
21991  A        21999  1153.977778
21992  A        22000  1578.060000

[21993 rows x 3 columns]


In [7]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)

stated_arpu
tenure_days
complaint_count


In [8]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    f_test(B[col], A[col])

stated_arpu
F = 0.9374  p = 0.0006932
tenure_days
F = 0.9762  p = 0.2061
complaint_count
F = 0.9765  p = 0.2128


In [9]:
a = pre[pre.g == 'A'].m
b = pre[pre.g == 'B'].m
a

1         256.590909
2        1914.133333
5         291.348077
6        4793.411111
13       4313.211111
            ...     
21982    1001.359259
21988    6681.156863
21989    1460.978571
21991    1153.977778
21992    1578.060000
Name: m, Length: 10990, dtype: float64

In [10]:
rev=q("""
select c.ab_group AS g,
coalesce(sum(r.amount_inr),0)as s
from customer_features c
left join recharges r on r.customer_id = c.customer_id
and r.recharge_date < '2025-01-01'
and r.recharge_date >= '2025-05-01'
group by c.customer_id 
""")
rev

,g,s
0,B,0
1,A,0
2,A,0
3,B,0
4,B,0
...,...,...
21995,A,0
21996,A,0
21997,B,0
21998,A,0


In [11]:
A = rev[rev.g== 'A'].s
B = rev[rev.g== 'B'].s

f_test(B,A)

z_mean2(B,A)

F = nan  p = nan
z = nan  p = nan


C:\Users\muham\AppData\Local\Temp\ipykernel_14628\733737059.py:9: RuntimeWarning: invalid value encountered in scalar divide
  f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
C:\Users\muham\AppData\Local\Temp\ipykernel_14628\733737059.py:18: RuntimeWarning: invalid value encountered in scalar divide
  z = (np.mean(x) - np.mean(y)) / se         # gap, in se units


In [12]:
q("""
select ab_group, count(*) as n,
sum(device_type = 'Smartphone')as smart,
sum (connection_type = 'Prepaid') as prepaid
from customer_features group by ab_group;
""")

,ab_group,n,smart,prepaid
0,A,10996,9417,8575
1,B,11004,9554,8574


In [13]:
#smart phone
z_prop2(9554,11004,9417,10996)

z = 2.5457  p = 0.01091


In [14]:
#prepaid
z_prop2(8574,11004,8575,10996)

z = -0.1177  p = 0.9063


In [15]:
off = q("""
SELECT c.ab_group AS g, u.customer_id, 
AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date >= '2025-05-01'
GROUP BY c.ab_group, u.customer_id;
""")
off

,g,customer_id,m
0,A,3,1758.756250
1,A,6,339.763333
2,A,7,5363.175000
3,A,14,3828.633333
4,A,15,580.526316
...,...,...,...
18443,B,21986,1049.761905
18444,B,21991,1922.758621
18445,B,21992,1033.139130
18446,B,21993,1936.235714


In [16]:
A=off[off.g=='A'].m
B=off[off.g=='B'].m

In [17]:
q("""
SELECT ab_group, COUNT(*) AS n,
       SUM(is_churned) AS all_churn,
       SUM(churn_date >= '2025-05-01') AS may_on
FROM   customer_features GROUP BY ab_group;
""")

,ab_group,n,all_churn,may_on
0,A,10996,2587,905
1,B,11004,2639,914


In [18]:
# Claim :the offer raised revenue
# H₀ :both groups paid the same on average
# H₁ :group B paid more
# Tail : one-tailed — the company said so in advance
Cev = q("""
SELECT c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r ON r.customer_id=c.customer_id
    AND r.recharge_date >= '2025-05-01'
GROUP BY c.customer_id;
""")

Cev

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


In [19]:
a=Cev[Cev.g=='A'].s
b=Cev[Cev.g=='B'].s
a,b

(1           0
 2        1665
 5         396
 6        3576
 13       3529
          ... 
 21989       0
 21995    3587
 21996       0
 21998       0
 21999       0
 Name: s, Length: 10996, dtype: int64,
 0          0
 3          0
 4          0
 7          0
 8        349
         ... 
 21991    992
 21992    349
 21993      0
 21994      0
 21997    209
 Name: s, Length: 11004, dtype: int64)

In [20]:
f_test(b,a)

F = 0.9697  p = 0.1068


In [21]:
stats.ttest_ind(a, b, equal_var=False, alternative='greater')

TtestResult(statistic=np.float64(0.5926613508590068), pvalue=np.float64(0.2767069527137275), df=np.float64(21992.29505989245))

In [22]:
# Claim : prepaid and postpaid subscribers churn at different rates
# H₀ :both have the same true churn rate
#H₁ :the two rates differ
# Claim : prepaid and postpaid subscribers churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q(""" 
SELECT connection_type, COUNT(*) AS grp,
       COUNT(*) AS n,SUM(is_churned) AS k
FROM customer_features
GROUP BY connection_type;
""")
z_prop2(4441,17149,785,4851)
# p-value<0.05 so its significant
# hence reject ho

z = 14.0364  p = 9.333e-45


In [23]:
# Claim :subscribers with an open ticket churn at a different rate from those who never complained
# H₀ :both have the same true churn rate
# H₁ :the two rates differ
q("""
SELECT CASE WHEN complaint_count = 0 THEN 'never'
WHEN unresolved_count = 0 THEN 'all closed'
ELSE 'open' END AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY grp;
""")
z_prop2(1834,6680,1328,6631)
# p-value<0.05 so its significant
# hence reject ho

z = 10.0684  p = 7.623e-24


In [24]:
# Claim :year-0 and year-3 subscribers churn at different rates
# H₀ :they share one true churn rate
# H₁ :they do not
q("""
SELECT tenure_years AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY tenure_years;
""")
(1328,4610,632,3374)
# p-value<0.05 soz_prop2 its significant
# hence reject ho

(1328, 4610, 632, 3374)

In [25]:
# Claim :Himachal Pradesh and Punjab churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT circle_name AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY circle_name;
""")
z_prop2(417,1247,77,558)
# p-value<0.05 soz_prop2 its significant
# hence reject ho

z = 8.6494  p = 5.176e-18


In [26]:
# Claim :smartphone and feature-phone users churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ
q("""
SELECT device_type AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY device_type;
""")
z_prop2(4507,18971,719,3029)
# p-value > 0.05 soz_prop2 its not significant
# hence failed to reject ho

z = 0.0241  p = 0.9807


In [27]:
# Claim :gender and device type each change the data a subscriber uses
# H₀ :both groups have the same true average daily data
# H₁ :the two averages differ

gen = q("""
SELECT c.gender, c.device_type,
u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id;
""")
dev = q("""
SELECT c.gender, c.device_type,
AVG(u.data_mb) AS m
from usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY c.customer_id;
""")

In [28]:
male=gen[gen.gender=='Male'].m
female=gen[gen.gender=='Female'].m
male
f_test(female,male)
# p value <0.05
# we can for z test
z_mean2(female,male)
# p value < 0.05 so significant

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [29]:
male=gen[gen.gender=='Male'].m
female=gen[gen.gender=='Female'].m
male
f_test(female,male)
# p value <0.05
# we can for z test
z_mean2(female,male)
# p value < 0.05 so significant

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [30]:
fp=dev[dev.device_type=='Feature Phone'].m
sp=dev[dev.device_type=='Smartphone'].m
f_test(fp,sp)

# p value <0.05 hence we can go for z test

z_mean2(fp,sp)
# hece reject ho

F = 0.0268  p = 0
z = -119.2400  p = 0
